# Fase 0 (parte 2) — Decisiones: corregir o excluir

Continuación de `tass_fase0_auditoria.ipynb`. El enunciado exige, para cada característica con alerta: **"corrijan la característica o justifiquen por qué la excluyen"**. Este notebook documenta esa decisión para las 6 características marcadas, y valida con evidencia real los 2 fixes aplicados en `logic/text_processing.py`.


## Resumen de decisiones

| Feature | Decisión | Justificación |
|---|---|---|
| `label_hashtag` | **Corregir** | Bug de typo (`[HASTAG]` vs. `'hashtag'`). La información sí existe en el corpus; se corrige el placeholder en `text_processing.py`. |
| `label_emoji` | **Corregir** | `proper_encoding()` destruía los emojis antes de que el regex de `[EMOJI]` pudiera detectarlos. Se reordena el pipeline: el reemplazo de emoji ahora ocurre sobre el texto crudo, antes de `proper_encoding()`. |
| `weighted_position` | **Excluir** | Aun corrigiendo el bug de `.index()` (que usa la posición del primer duplicado en vez de la real), la fórmula sigue midiendo esencialmente la posición relativa de las palabras, acoplada matemáticamente al largo del texto (r=0.72 con n° de tokens). No se justifica la complejidad para una señal marginal. |
| `weighted_normalized` | **Excluir** | Se reduce algebraicamente a `(n+1)/2`: depende solo de `n`, nunca del contenido (r=0.98). |
| `label_word` | **Excluir** | Correlación 0.99 con el número de tokens — es, en la práctica, un sinónimo de la longitud. El propio BoW/TF-IDF ya captura implícitamente esta información (suma de conteos). |
| `lexical_diversity` | **Reemplazar por una versión corregida** | El TTR (type-token ratio) es conocido en la literatura de PLN por depender de la longitud (r=-0.83 aquí). En vez de solo excluirla, se sustituye por **MATTR** (Moving-Average TTR, ventana móvil), que corrige justamente ese sesgo — y se cuenta como una de las características nuevas de la Fase 1. |

**Nota aparte (no forma parte de los 3 chequeos oficiales, pero es relevante):** `adverb_all` es una combinación lineal exacta de las otras 5 categorías de adverbio ya existentes. Se recomienda excluirla de la representación final por redundancia/colinealidad, aunque no viole ninguno de los 3 criterios del enunciado.


## Validación del fix: `label_hashtag`

In [2]:
import sys
sys.path.insert(0, '..')

import importlib
import logic.text_processing
importlib.reload(logic.text_processing)  # asegura que se cargue el codigo corregido, no una version en cache

from logic.text_processing import TextProcessing
from logic.feature_extraction import FeatureExtraction
import pandas as pd

fe = FeatureExtraction(lang='es')

tweet_hashtag = "Se ha terminado #Rio2016 Lamentablemente no arriendo"
resultado = TextProcessing.transformer(tweet_hashtag)
print("Resultado:", repr(resultado))
assert 'hashtag' in resultado.split(), "El fix deberia producir el token 'hashtag'"
assert 'hastag' not in resultado.split(), "Ya no deberia aparecer el token roto 'hastag'"
print("OK: fix de hashtag confirmado")


Resultado: 'se ha terminado hashtag lamentablemente no arriendo'
OK: fix de hashtag confirmado


## Validación del fix: `label_emoji`

In [3]:
tweet_emoji = "Que alegria!! 😊 🎉 estoy feliz"
resultado = TextProcessing.transformer(tweet_emoji)
print("Resultado:", repr(resultado))
assert 'emoji' in resultado.split(), "El fix deberia producir el token 'emoji'"
print("OK: fix de emoji confirmado")


Resultado: 'que alegria emoji emoji estoy feliz'
OK: fix de emoji confirmado


## Validación a escala: ¿cuánto cambia en todo el corpus?

Antes de los fixes, ambas features daban 0% de activación en los 1008 tweets del train. Verificamos cuántos tweets activan cada una ahora.


In [4]:
df = pd.read_csv('../data/tass/tass2018_es_train.csv')
texts = df['content'].dropna().tolist()

activa_hashtag = 0
activa_emoji = 0
total = 0

for t in texts:
    proc = TextProcessing.transformer(t)
    if not proc:
        continue
    vec = fe.get_features_lexical(proc)
    if vec is None:
        continue
    total += 1
    if vec[4] > 0:
        activa_hashtag += 1
    if vec[5] > 0:
        activa_emoji += 1

print(f"De {total} tweets en train:")
print(f"  label_hashtag se activa en {activa_hashtag} tweets ({100*activa_hashtag/total:.1f}%) -- antes del fix: 0 (0.0%)")
print(f"  label_emoji se activa en {activa_emoji} tweets ({100*activa_emoji/total:.1f}%) -- antes del fix: 0 (0.0%)")


De 1008 tweets en train:
  label_hashtag se activa en 72 tweets (7.1%) -- antes del fix: 0 (0.0%)
  label_emoji se activa en 17 tweets (1.7%) -- antes del fix: 0 (0.0%)


## Features finales para la Fase 2 (unión BoW + léxico)

Con las decisiones de arriba, el vector de características léxicas que se usará de aquí en adelante **excluye**: `weighted_position`, `weighted_normalized`, `label_word`, `lexical_diversity` (original), `adverb_all`.

`lexical_diversity` se reemplaza por `lexical_diversity_mattr` (implementada en la Fase 1, junto con las demás características nuevas del equipo).


In [5]:
NOMBRES_FEATURES_ORIGINALES = [
    'weighted_position', 'weighted_normalized', 'label_mention', 'label_url', 'label_hashtag',
    'label_emoji', 'label_retweets', 'lexical_diversity', 'label_word',
    'first_person_singular', 'second_person_singular', 'third_person_singular',
    'first_person_plurar', 'second_person_plurar', 'third_person_plurar',
    'avg_word', 'kur_word', 'skew_word',
    'adverb_neg', 'adverb_time', 'adverb_place', 'adverb_mode', 'adverb_cant', 'adverb_all',
    'adjetives_neg', 'adjetives_pos', 'who_general', 'who_male', 'who_female'
]

EXCLUIDAS = {'weighted_position', 'weighted_normalized', 'label_word', 'lexical_diversity', 'adverb_all'}

FEATURES_VALIDADAS_FASE0 = [f for f in NOMBRES_FEATURES_ORIGINALES if f not in EXCLUIDAS]
print(f"Features originales: {len(NOMBRES_FEATURES_ORIGINALES)}")
print(f"Features excluidas: {len(EXCLUIDAS)} -> {sorted(EXCLUIDAS)}")
print(f"Features que pasan la auditoria de la Fase 0: {len(FEATURES_VALIDADAS_FASE0)}")


Features originales: 29
Features excluidas: 5 -> ['adverb_all', 'label_word', 'lexical_diversity', 'weighted_normalized', 'weighted_position']
Features que pasan la auditoria de la Fase 0: 24
